## Módulo 5. Análisis de Calidad del Aire con técnicas de interpolación espacial

In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
gdf = gpd.read_file('st_rb_2019_nox.gpkg')
gdf['NOx_avg19'].isna().any()
gdf.crs

In [ ]:
gdf = gdf.dropna(subset=['NOx_avg19'])
gdf['NOx_avg19'].isna().any()
gdf = gdf.to_crs(epsg=3035)
print(gdf.crs)

In [ ]:
x = gdf.geometry.x.to_numpy()
y = gdf.geometry.y.to_numpy()
coords = np.column_stack([x, y])
values = gdf['NOx_avg19'].to_numpy(dtype=np.float32)
vmin, vmax = np.percentile(values, [5, 95])

In [ ]:
# Creacion de rejilla con Verde 
import verde as vd
region = vd.get_region((x, y))
region = vd.pad_region(region, 50_000)
print(region)

east, north = vd.grid_coordinates(
    region=region,
    spacing=10_000
)

query_coords = np.column_stack([east.ravel(), north.ravel()])

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(east, north, s=10, color='red')
plt.gca().set_aspect("equal")
plt.xlabel("X / Easting")
plt.ylabel("Y / Northing")
plt.title("Grilla creada con Verde")
plt.show()

### **Ponderación por Distancia Inversa (IDW)**
La **Ponderación por Distancia Inversa** (*Inverse Distance Weighting*, IDW) es un método determinístico de interpolación espacial. Estima el valor de una variable en una ubicación desconocida a partir de los valores observados en puntos cercanos, asignando mayor peso a los puntos más próximos y menor peso a los más alejados.

Para una ubicación desconocida \(x_0\), el valor estimado se calcula como:

$$
\hat{z}(x_0) = \frac{\sum_{i=1}^{n} w_i z_i}{\sum_{i=1}^{n} w_i}
$$

donde \(z_i\) es el valor observado en el punto \(i\), y \(w_i\) es el peso asignado según la distancia al punto desconocido:

$$
w_i = \frac{1}{d_i^p}
$$

con:

$$
d_i = ||x_0 - x_i||
$$

donde:

- $z_i$ es el valor observado en el punto conocido $i$.
- $w_i$ es el peso asignado a ese punto.
- $d_i$ es la distancia euclidiana entre la ubicación desconocida $x_0$ y el punto conocido $x_i$.
- $p$ es el parámetro de potencia que controla cuánto disminuye la influencia con la distancia.

Valores altos de $p$ hacen que los puntos cercanos tengan mucha más influencia, mientras que valores bajos generan una superficie más suavizada.


En Python, el cálculo se realizará usando `scipy.spatial.cKDTree`, que permite buscar eficientemente los puntos vecinos más cercanos, junto con `NumPy` para calcular los pesos IDW y obtener los valores interpolados sobre la rejilla.


In [ ]:
from scipy.spatial import cKDTree

In [ ]:
def idw_interpolation(coords, values, query_coords, k=12, radius=np.inf, power=2, output_shape=None):
    tree = cKDTree(coords)
    distances, indices = tree.query(query_coords, k=k, distance_upper_bound=radius, workers=-1)
    valid = np.isfinite(distances)
    mask_min = valid.sum(axis=1) >= 3
    #print(distances)
    #print(mask_min)

    weights = 1 / np.maximum(distances, 1e-12) ** power
    weights[~valid] = 0

    safe_indices = np.where(indices < len(values), indices, 0)
    #print(indices.min(), indices.max())
    #print(len(values))
    neighbor_values = values[safe_indices]

    numerator = np.sum(weights * neighbor_values, axis=1)
    denominator = np.sum(weights, axis=1)
    result = np.full(query_coords.shape[0], np.nan)
    result[mask_min] = numerator[mask_min] / denominator[mask_min]

    return result.reshape(output_shape)

idw_surface = idw_interpolation(
    coords=coords, 
    values=values, 
    query_coords=query_coords,
    k=12,
    radius=250_000, 
    power=3,
    output_shape=east.shape
    )

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
mesh = ax.pcolormesh(
    east, 
    north, 
    idw_surface, 
    shading='auto', 
    cmap='viridis', 
    vmin=vmin, 
    vmax=vmax 
    )
ax.scatter(
    x,
    y,
    c=values,
    cmap='viridis',
    edgecolor='black',
    s=20,
    linewidths=0.5,
    vmax=vmax,
    vmin=vmin
)
fig.colorbar(mesh, ax=ax, label='NOx_avg19')
ax.set_title('Interpolación IDW de NOx_avg19')
ax.set_aspect('equal')

plt.show()

### **Red irregular triangulada (TIN)**
La **Red Irregular Triangulada** (*Triangulated Irregular Network*, TIN) es un método de interpolación espacial que representa la superficie mediante una red de triángulos no superpuestos construidos a partir de los puntos conocidos. Normalmente se utiliza una **triangulación de Delaunay**, que evita triángulos muy alargados y genera una malla adecuada para interpolar localmente.

El valor en una ubicación desconocida $P(x, y)$ se estima mediante **interpolación lineal** dentro del triángulo que contiene ese punto. Si los vértices del triángulo son $P_1$, $P_2$ y $P_3$, con valores conocidos $z_1$, $z_2$ y $z_3$, entonces:

$$
z(x, y) = \lambda_1 z_1 + \lambda_2 z_2 + \lambda_3 z_3
$$

donde $\lambda_1$, $\lambda_2$ y $\lambda_3$ son las coordenadas baricéntricas del punto $P(x, y)$ dentro del triángulo. Estas coordenadas cumplen:

$$
\lambda_1 + \lambda_2 + \lambda_3 = 1
$$

$$
\lambda_i \geq 0, \quad i = 1, 2, 3
$$

También pueden interpretarse como proporciones de área:

$$
\lambda_1 = \frac{A_1}{A}, \qquad
\lambda_2 = \frac{A_2}{A}, \qquad
\lambda_3 = \frac{A_3}{A}
$$

donde $A$ es el área total del triángulo $P_1P_2P_3$, y $A_1$, $A_2$ y $A_3$ son las áreas de los subtriángulos formados por el punto $P(x, y)$ y los lados opuestos a cada vértice.

En Python, esta interpolación se realizará con `scipy.interpolate.LinearNDInterpolator`, que construye internamente una triangulación de Delaunay sobre los puntos conocidos y calcula valores interpolados mediante interpolación lineal dentro de cada triángulo.



In [ ]:
from scipy.interpolate import LinearNDInterpolator

In [ ]:
tin_interpolator = LinearNDInterpolator(coords, values)

tin_surface = tin_interpolator(east, north)
levels = np.linspace(vmin, vmax, 7)
levels

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
mesh = ax.pcolormesh(
    east, 
    north, 
    tin_surface, 
    shading='auto', 
    cmap='viridis', 
    vmin=vmin, 
    vmax=vmax
    )

ax.scatter(
    x,
    y,
    c=values,
    cmap='viridis',
    edgecolor='black',
    s=20,
    linewidths=0.5,
    vmax=vmax,
    vmin=vmin
)
cntr = ax.contour(
    east,
    north,
    tin_surface,
    levels=levels,
    colors='white',
    linewidths=0.5,
    alpha=0.8
)

fig.colorbar(mesh, ax=ax, label='NOx_avg19')
ax.set_title('Interpolación TIN de NOx_avg19')
ax.clabel(cntr, inline=True, fontsize=6)
ax.set_aspect('equal')
plt.show()

### **Interpolación Kriging**
La **interpolación Kriging** es un método geoestadístico que estima el valor de una variable en una ubicación desconocida considerando tanto la distancia entre puntos como la **autocorrelación espacial** de los datos. A diferencia de IDW, los pesos no dependen solo de la distancia, sino también de la estructura espacial modelada mediante el **variograma**.

El valor estimado en una ubicación desconocida $x_0$ se calcula como una combinación ponderada de los valores observados:

$$
Z^*(x_0) = \sum_{i=1}^{n} \lambda_i Z(x_i)
$$

donde:

- $Z^*(x_0)$ es el valor estimado en la ubicación desconocida $x_0$.
- $Z(x_i)$ es el valor observado en la ubicación conocida $x_i$.
- $\lambda_i$ es el peso asignado al punto $i$ por el sistema de Kriging.
- $n$ es el número de puntos utilizados en la estimación.


La dependencia espacial se describe mediante el **semivariograma**, que mide cómo cambia la diferencia entre valores según la distancia que separa los puntos:

$$
\gamma(h) = \frac{1}{2N(h)} \sum_{i=1}^{N(h)} \left[ Z(x_i) - Z(x_i + h) \right]^2
$$

donde:

- $\gamma(h)$ es la semivarianza para una distancia $h$.
- $N(h)$ es el número de pares de puntos separados aproximadamente por la distancia $h$.
- $Z(x_i)$ y $Z(x_i + h)$ son los valores observados en dos ubicaciones separadas por $h$.

Los principales parámetros del variograma son:

- **Nugget** $(C_0)$: variabilidad a distancias muy pequeñas o error de medida.
- **Sill** $(C_0 + C)$: valor máximo o meseta que alcanza el variograma.
- **Range** $(a)$: distancia a partir de la cual los puntos dejan de estar espacialmente correlacionados.

Para obtener los pesos $\lambda_i$, Ordinary Kriging resuelve un sistema lineal basado en el variograma. En forma matricial:

$$
\begin{bmatrix}
\gamma(x_1, x_1) & \gamma(x_1, x_2) & \cdots & \gamma(x_1, x_n) & 1 \\
\gamma(x_2, x_1) & \gamma(x_2, x_2) & \cdots & \gamma(x_2, x_n) & 1 \\
\vdots & \vdots & \ddots & \vdots & \vdots \\
\gamma(x_n, x_1) & \gamma(x_n, x_2) & \cdots & \gamma(x_n, x_n) & 1 \\
1 & 1 & \cdots & 1 & 0
\end{bmatrix}
\begin{bmatrix}
\lambda_1 \\
\lambda_2 \\
\vdots \\
\lambda_n \\
\mu
\end{bmatrix}
=
\begin{bmatrix}
\gamma(x_1, x_0) \\
\gamma(x_2, x_0) \\
\vdots \\
\gamma(x_n, x_0) \\
1
\end{bmatrix}
$$

donde $\gamma(x_i, x_j)$ representa la semivarianza entre dos puntos conocidos $x_i$ y $x_j$, $\gamma(x_i, x_0)$ representa la semivarianza entre un punto conocido y la ubicación a estimar $x_0$, y $\mu$ es el multiplicador de Lagrange que garantiza la condición:

$$
\sum_{i=1}^{n} \lambda_i = 1
$$


En Python, la interpolación Kriging se realizará con `pykrige.ok.OrdinaryKriging`, que permite ajustar un modelo de variograma y generar tanto la superficie interpolada como la varianza de predicción asociada.


In [ ]:
# type: ignore
import scipy.stats as stats
from pykrige.ok import OrdinaryKriging

In [ ]:
log_values = np.log1p(values)
vmin_log, vmax_log = np.percentile(log_values, [5, 95])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(values, bins=30)
axes[0].set_title("Datos originales")
axes[0].set_xlabel("NOx")
axes[0].set_ylabel("Frecuencia")

axes[1].hist(log_values, bins=30)
axes[1].set_title("distribución normal con log1p()")
axes[1].set_xlabel("log1p(NOx)")
axes[1].set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

In [ ]:
# Datos originales
stat, p = stats.shapiro(values)
skew = stats.skew(values)
kurt = stats.kurtosis(values)
print(f"Original      : Shapiro p={p:.4f} | Skew={skew:.2f} | Kurt={kurt:.2f}")

# Tras log-transform
stat2, p2 = stats.shapiro(log_values)
print(f"Log-transform : Shapiro p={p2:.4f}")

In [ ]:
# PyKrige necesita ejes 1D
grid_east = east[0, :]
grid_north = north[:, 0]

OK = OrdinaryKriging(
    x,
    y,
    log_values,
    variogram_model="spherical",
    nlags=20,
    verbose=True,
    enable_plotting=True
)

# Predicción en espacio log
z_log, ss_log = OK.execute("grid", grid_east, grid_north)

# Transformar de vuelta a espacio original
z_log_array = np.ma.filled(z_log, fill_value=np.nan)
ss_log_array = np.ma.filled(ss_log, fill_value=np.nan)

z_original = np.expm1(z_log_array)

In [ ]:
# Visualización
fig, axes = plt.subplots(1, 3, figsize=(22, 7), sharex=True, sharey=True)

# Mapa 1: Kriging en espacio log
im1 = axes[0].pcolormesh(
    east,
    north,
    z_log_array,
    shading="auto",
    cmap="viridis",
    vmin=vmin_log,
    vmax=vmax_log
)

axes[0].scatter(
    x,
    y,
    c="white",
    s=12,
    edgecolors="black",
    alpha=0.5,
    zorder=5
)

axes[0].set_title(
    "Ordinary Kriging en escala log",
    fontsize=13
)

fig.colorbar(
    im1,
    ax=axes[0],
    label="log1p(NOx)"
)

# Mapa 2: Incertidumbre del Kriging en escala log

im2 = axes[1].pcolormesh(
    east,
    north,
    ss_log_array,
    shading="auto",
    cmap="magma"
)

axes[1].scatter(
    x,
    y,
    c="white",
    s=12,
    edgecolors="black",
    alpha=0.5,
    zorder=5
)

axes[1].set_title(
    "Incertidumbre Kriging en escala log",
    fontsize=13
)

fig.colorbar(
    im2,
    ax=axes[1],
    label="Varianza de log1p(NOx)"
)

# Mapa 3: Predicción final en escala original

im3 = axes[2].pcolormesh(
    east,
    north,
    z_original,
    shading="auto",
    cmap="viridis",
    vmin=vmin,
    vmax=vmax
)

axes[2].scatter(
    x,
    y,
    c="white",
    s=12,
    edgecolors="black",
    alpha=0.5,
    zorder=5
)

axes[2].set_title(
    "Ordinary Kriging en escala original",
    fontsize=13
)

fig.colorbar(
    im3,
    ax=axes[2],
    label="NOx_avg19"
)

for ax in axes:
    ax.set_aspect("equal")
    ax.set_xlabel("Este (m)")
    ax.set_ylabel("Norte (m)")

plt.suptitle(
    "Ordinary Kriging con transformación log1p",
    fontsize=15,
    y=1.02
)

plt.tight_layout()
plt.show()

In [ ]:
errors_log = []
errors_orig = []
std_errors_log = []

print("\nEjecutando validación cruzada LOO...")

for i in range(len(log_values)):
    x_loo = np.delete(x, i)
    y_loo = np.delete(y, i)
    v_loo = np.delete(log_values, i)

    ok_loo = OrdinaryKriging(
        x_loo, y_loo, v_loo,
        variogram_model="spherical",
        nlags=15,
        verbose=False,
        enable_plotting=False
    )

    pred_log, ss_log = ok_loo.execute(
        "points",
        np.array([x[i]]),
        np.array([y[i]])
    )

    pred_log = float(pred_log[0])
    ss_log = float(ss_log[0])

    err_log = pred_log - log_values[i]
    errors_log.append(err_log)
    errors_orig.append(np.expm1(pred_log) - values[i])

    if ss_log > 0:
        std_errors_log.append(err_log / np.sqrt(ss_log))

errors_log = np.array(errors_log)
errors_orig = np.array(errors_orig)
std_errors_log = np.array(std_errors_log)

print("\n=== LOO — espacio log ===")
print(f"ME   : {errors_log.mean():.4f}")
print(f"RMSE : {np.sqrt(np.mean(errors_log**2)):.4f}")

print("\n=== LOO — errores estandarizados ===")
print(f"ME est.   : {std_errors_log.mean():.4f}")
print(f"RMSE est. : {np.sqrt(np.mean(std_errors_log**2)):.4f}  (objetivo ≈ 1)")

print("\n=== LOO — espacio original µg/m³ ===")
print(f"ME   : {errors_orig.mean():.4f}")
print(f"RMSE : {np.sqrt(np.mean(errors_orig**2)):.4f}")
print(f"MAE  : {np.mean(np.abs(errors_orig)):.4f}")


### **Práctica: Análisis de Calidad del Aire**

#### Descripción

Analiza datos europeos de calidad del aire utilizando técnicas de interpolación espacial. Los datos deberán descargarse desde el portal de la **European Environment Agency (EEA)**: [European air quality data (interpolated data and station points)](https://www.eea.europa.eu/en/datahub/datahubitem-view/b51e1091-4459-4a1e-8dbc-dd7a30949b90).

Este conjunto de datos contiene información espacial de contaminantes atmosféricos como `NO2`, `NOx`, `O3`, `PM10`, `PM25` y `POD6`, incluyendo puntos de estaciones de medida y superficies interpoladas. Para esta práctica se deberán seleccionar **dos contaminantes diferentes a `NOx`**, por ejemplo `NO2` y `O3` con el objetivo de aplicar y comparar tres métodos de interpolación espacial:

- **Ponderación por Distancia Inversa** (`IDW`)
- **Red Irregular Triangulada** (`TIN`)
- **Kriging ordinario**

El análisis debe incluir la carga de datos, la preparación de las coordenadas, la generación de una rejilla espacial, la aplicación de los tres métodos de interpolación y la interpretación de los resultados obtenidos para cada contaminante.

#### Actividades

1. Descargar los datos de calidad del aire desde el portal de la **European Environment Agency (EEA)**.
2. Seleccionar dos contaminantes diferentes a `NOx`, por ejemplo `NO2` y `O3`.
3. Cargar los datos de estaciones de medida utilizando `geopandas`.
4. Revisar el sistema de referencia espacial y reproyectar los datos si es necesario.
5. Preparar las coordenadas de los puntos de medición y los valores de concentración de cada contaminante.
6. Crear una rejilla espacial sobre el área de estudio utilizando `verde`.
7. Aplicar interpolación **IDW** utilizando `scipy.spatial.cKDTree` y `NumPy`.
8. Aplicar interpolación **TIN** utilizando `scipy.interpolate.LinearNDInterpolator`.
9. Aplicar **Kriging ordinario** utilizando `pykrige.ok.OrdinaryKriging`.
10. Generar mapas comparativos para cada contaminante y método de interpolación.
11. Exportar las gráficas generadas a una carpeta `outputs/`.

#### Entregable

Carpeta de datos `outputs/` y Notebook `.ipynb` con el desarrollo completo de la práctica, las gráficas ejecutadas y un apartado final de interpretación.

La carpeta `outputs/` debe incluir las gráficas exportadas, por ejemplo:

- `no2_idw.png`
- `no2_tin.png`
- `no2_kriging.png`
- `o3_idw.png`
- `o3_tin.png`
- `o3_kriging.png`

En la interpretación final se debe explicar:

- Qué diferencias se observan entre los métodos `IDW`, `TIN` y `Kriging ordinario`.
- Qué zonas presentan mayores y menores concentraciones para cada contaminante.
- Qué método genera superficies más suaves o más abruptas.
- Qué ventajas y limitaciones se observan en cada método.